# Fine-Tuning

As we saw in yesterday's class, RAGs + Vector DB's do offer some promise in improving LLM responses, but for small models like `distilgpt2`, it still isn't sufficient in generating "good" answers.

Outside of the semantic search on vector databases, the RAG we implemented yesterday could honestly just be boiled down to more "advanced prompting." If we want to sufficiently influence the models responses, we must modify the *weights* of the model, and give it feedback as to which responses are "good" or "bad."

If we think back to supervised machine learning, we can envision a framework which we could potentially apply to our large language models. Instead of augmenting the user prompt, why don't we "train" the models weights using a dataset of prompts & targets. We could even split this dataset into a training & testing set, which would allow us to measure the "accuracy" of the model by measuring how "distant" or "close" a models response is to our intended output.

**An example training set for customer service responses**
```bash
{
    {
        "prompt": "Help! My wifi isn't working."
        "answer": "I'm sorry to hear about this issue! While we wait for this issue to resolve, please utilize the free hotspot provided with your internet package."
    },
    {
        "prompt": "How do I sign up?"
        "answer": "Please visit our website at wwww.xyz.com to create an account."
    }
}
```

This describes the essence of fine-tuning. Follow along with the code below to find out more.

In [55]:
!pip install transformers datasets accelerate

## Fine-Tuning

Fine-tuning means taking an already pre-trained model (like distilgpt2, which has already learned English syntax and general knowledge) and training it further on domain-specific data so it adapts to your use case.

Think of it like this:
* Pretraining: Learning how to read & write in general.  
* Fine-tuning: Learning to write restaurant reviews in your specific tone.  

We use fine-tuning when we want our model to:
* adopt a specific style, tone, or format,
* use specialized vocabulary not well-covered by the base model,
* behave consistently.

As we saw yesterday, the default `distilgpt2` model has poor reasoning capabilities beyond a few tokens.

In [56]:
from transformers import pipeline

model_id = "distilgpt2"

pipe = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype="auto",
    device_map="auto"
)

prompt = "Who was the second US president?"

out = pipe(prompt, max_new_tokens=5, do_sample=True, temperature=1.0)
print(out[0]["generated_text"])

Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Who was the second US president?



Now


## Fine-Tuning Data

Fine-tuning data should be:

* Representative: Contains the type of inputs & outputs you expect at inference.
* Clean: Avoids spelling mistakes, contradictions, or inconsistent formats.
* Balanced: Covers multiple variations of your task.

As we saw in the example above, we often utilize input-output pairs to create our dataset.

**Input-Output Pairs**
```bash
{
    "prompt": "Summarize: The cat chased the mouse into the garden.", 
    "completion": "The cat pursued the mouse into the garden."
}
```

However, for GPT-style models, we can also utilize the text-continuation format
**Text-Continuation Format**
```bash
{
    "text": "Summarize: The cat chased the mouse into the garden.\nThe cat pursued the mouse into the garden."
}
```

Let's create an Input-Output Pairs dataset to tune `distilgpt2`.

In [145]:
# create a list of text to fine-tune our model
train_data = [
    {"input": "Q: Who was the 1st U.S. president?\n", "labels": "Q: Who was the 1st U.S. president?\nA: George Washington."},
    {"input": "Q: What years did George Washington serve as president?\n", "labels": "Q: What years did George Washington serve as president?\nA: 1789–1797."},
    {"input": "Q: What political party did George Washington belong to?\n", "labels": "Q: What political party did George Washington belong to?\nA: None (independent/no formal party)."},
    {"input": "Q: Who was the 2nd U.S. president?\n", "labels": "Q: Who was the 2nd U.S. president?\nA: John Adams."},
    {"input": "Q: What party was John Adams?\n", "labels": "Q: What party was John Adams?\nA: Federalist."},
    {"input": "Q: Who was the 3rd U.S. president?\n", "labels": "Q: Who was the 3rd U.S. president?\nA: Thomas Jefferson."},
    {"input": "Q: What party was Thomas Jefferson?\n", "labels": "Q: What party was Thomas Jefferson?\nA: Democratic-Republican."},
    {"input": "Q: Who was the 4th U.S. president?\n", "labels": "Q: Who was the 4th U.S. president?\nA: James Madison."},
    {"input": "Q: Who was the 5th U.S. president?\n", "labels": "Q: Who was the 5th U.S. president?\nA: James Monroe."},
    {"input": "Q: What was the Monroe Doctrine mainly about?\n", "labels": "Q: What was the Monroe Doctrine mainly about?\nA: Opposing European colonization in the Americas."},
    {"input": "Q: Who was the 7th U.S. president?\n", "labels": "Q: Who was the 7th U.S. president?\nA: Andrew Jackson."},
    {"input": "Q: Which war made Andrew Jackson nationally famous before his presidency?\n", "labels": "Q: Which war made Andrew Jackson nationally famous before his presidency?\nA: War of 1812 (Battle of New Orleans)."},
    {"input": "Q: Who was the 11th U.S. president?\n", "labels": "Q: Who was the 11th U.S. president?\nA: James K. Polk."},
    {"input": "Q: Which conflict occurred during James K. Polk’s presidency?\n", "labels": "Q: Which conflict occurred during James K. Polk’s presidency?\nA: Mexican–American War."},
    {"input": "Q: Who was the 16th U.S. president?\n", "labels": "Q: Who was the 16th U.S. president?\nA: Abraham Lincoln."},
    {"input": "Q: What party was Abraham Lincoln?\n", "labels": "Q: What party was Abraham Lincoln?\nA: Republican."},
    {"input": "Q: What years did Abraham Lincoln serve?\n", "labels": "Q: What years did Abraham Lincoln serve?\nA: 1861–1865."},
    {"input": "Q: Which major conflict occurred during Lincoln’s presidency?\n", "labels": "Q: Which major conflict occurred during Lincoln’s presidency?\nA: The American Civil War."},
    {"input": "Q: Who issued the Emancipation Proclamation?\n", "labels": "Q: Who issued the Emancipation Proclamation?\nA: Abraham Lincoln."},
    {"input": "Q: Who was the 18th U.S. president?\n", "labels": "Q: Who was the 18th U.S. president?\nA: Ulysses S. Grant."},
    {"input": "Q: Who was the 26th U.S. president?\n", "labels": "Q: Who was the 26th U.S. president?\nA: Theodore Roosevelt."},
    {"input": "Q: What years did Theodore Roosevelt serve?\n", "labels": "Q: What years did Theodore Roosevelt serve?\nA: 1901–1909."},
    {"input": "Q: Who was the 32nd U.S. president?\n", "labels": "Q: Who was the 32nd U.S. president?\nA: Franklin D. Roosevelt."},
    {"input": "Q: What years did Franklin D. Roosevelt serve?\n", "labels": "Q: What years did Franklin D. Roosevelt serve?\nA: 1933–1945."},
    {"input": "Q: Which two major crises defined Franklin D. Roosevelt’s presidency?\n", "labels": "Q: Which two major crises defined Franklin D. Roosevelt’s presidency?\nA: The Great Depression and World War II."},
    {"input": "Q: Who launched the New Deal programs?\n", "labels": "Q: Who launched the New Deal programs?\nA: Franklin D. Roosevelt."},
    {"input": "Q: Which president authorized the use of atomic bombs in WWII?\n", "labels": "Q: Which president authorized the use of atomic bombs in WWII?\nA: Harry S. Truman."},
    {"input": "Q: Who was the 34th U.S. president?\n", "labels": "Q: Who was the 34th U.S. president?\nA: Dwight D. Eisenhower."},
    {"input": "Q: What party was Dwight D. Eisenhower?\n", "labels": "Q: What party was Dwight D. Eisenhower?\nA: Republican."},
    {"input": "Q: Who was the 35th U.S. president?\n", "labels": "Q: Who was the 35th U.S. president?\nA: John F. Kennedy."},
    {"input": "Q: What years did John F. Kennedy serve?\n", "labels": "Q: What years did John F. Kennedy serve?\nA: 1961–1963."},
    {"input": "Q: Which crisis occurred in 1962 during JFK’s presidency?\n", "labels": "Q: Which crisis occurred in 1962 during JFK’s presidency?\nA: The Cuban Missile Crisis."},
    {"input": "Q: Who was the 36th U.S. president?\n", "labels": "Q: Who was the 36th U.S. president?\nA: Lyndon B. Johnson."},
    {"input": "Q: Which landmark civil rights laws were signed under Lyndon B. Johnson?\n", "labels": "Q: Which landmark civil rights laws were signed under Lyndon B. Johnson?\nA: Civil Rights Act of 1964 and Voting Rights Act of 1965."},
    {"input": "Q: Who was the 37th U.S. president?\n", "labels": "Q: Who was the 37th U.S. president?\nA: Richard Nixon."},
    {"input": "Q: Which major scandal led to a presidential resignation in 1974?\n", "labels": "Q: Which major scandal led to a presidential resignation in 1974?\nA: Watergate."},
    {"input": "Q: Who was the 38th U.S. president?\n", "labels": "Q: Who was the 38th U.S. president?\nA: Gerald Ford."},
    {"input": "Q: Who was the 39th U.S. president?\n", "labels": "Q: Who was the 39th U.S. president?\nA: Jimmy Carter."},
    {"input": "Q: Who was the 40th U.S. president?\n", "labels": "Q: Who was the 40th U.S. president?\nA: Ronald Reagan."},
    {"input": "Q: What years did Ronald Reagan serve?\n", "labels": "Q: What years did Ronald Reagan serve?\nA: 1981–1989."},
    {"input": "Q: Who was the 41st U.S. president?\n", "labels": "Q: Who was the 41st U.S. president?\nA: George H. W. Bush."},
    {"input": "Q: Which conflict occurred during George H. W. Bush’s presidency?\n", "labels": "Q: Which conflict occurred during George H. W. Bush’s presidency?\nA: The Gulf War (1990–1991)."},
    {"input": "Q: Who was the 42nd U.S. president?\n", "labels": "Q: Who was the 42nd U.S. president?\nA: Bill Clinton."},
    {"input": "Q: Who was the 43rd U.S. president?\n", "labels": "Q: Who was the 43rd U.S. president?\nA: George W. Bush."},
    {"input": "Q: Which major event defined early George W. Bush’s presidency?\n", "labels": "Q: Which major event defined early George W. Bush’s presidency?\nA: The September 11, 2001 attacks."},
    {"input": "Q: Who was the 44th U.S. president?\n", "labels": "Q: Who was the 44th U.S. president?\nA: Barack Obama."},
    {"input": "Q: What years did Barack Obama serve?\n", "labels": "Q: What years did Barack Obama serve?\nA: 2009–2017."},
    {"input": "Q: Which major health care law was enacted under Barack Obama?\n", "labels": "Q: Which major health care law was enacted under Barack Obama?\nA: The Affordable Care Act (ACA)."},
    {"input": "Q: Who was the 45th U.S. president?\n", "labels": "Q: Who was the 45th U.S. president?\nA: Donald Trump."},
    {"input": "Q: What years did Donald Trump serve?\n", "labels": "Q: What years did Donald Trump serve?\nA: 2017–2021."},
    {"input": "Q: Who is the 46th U.S. president?\n", "labels": "Q: Who is the 46th U.S. president?\nA: Joe Biden."},
    {"input": "Q: When did Joe Biden take office?\n", "labels": "Q: When did Joe Biden take office?\nA: January 20, 2021."},
    {"input": "Q: Which president served the longest time in office?\n", "labels": "Q: Which president served the longest time in office?\nA: Franklin D. Roosevelt."},
    {"input": "Q: Which amendment limits presidents to two terms?\n", "labels": "Q: Which amendment limits presidents to two terms?\nA: The 22nd Amendment."},
    {"input": "Q: Who was president during the Louisiana Purchase?\n", "labels": "Q: Who was president during the Louisiana Purchase?\nA: Thomas Jefferson."},
    {"input": "Q: Who was president when the Civil Rights Act of 1964 was signed?\n", "labels": "Q: Who was president when the Civil Rights Act of 1964 was signed?\nA: Lyndon B. Johnson."},
    {"input": "Q: Which president delivered the Gettysburg Address?\n", "labels": "Q: Which president delivered the Gettysburg Address?\nA: Abraham Lincoln."},
    {"input": "Q: Who was president during the Bay of Pigs invasion?\n", "labels": "Q: Who was president during the Bay of Pigs invasion?\nA: John F. Kennedy."},
    {"input": "Q: Which president signed NAFTA into law?\n", "labels": "Q: Which president signed NAFTA into law?\nA: Bill Clinton."},
    {"input": "Q: Who was president during the Iraq War invasion in 2003?\n", "labels": "Q: Who was president during the Iraq War invasion in 2003?\nA: George W. Bush."},
    {"input": "Q: Which president established the national parks system foundation with aggressive conservation policies?\n", "labels": "Q: Which president established the national parks system foundation with aggressive conservation policies?\nA: Theodore Roosevelt."},
    {"input": "Q: Who was president when the U.S. entered World War I?\n", "labels": "Q: Who was president when the U.S. entered World War I?\nA: Woodrow Wilson."},
    {"input": "Q: Who was president when the U.S. entered World War II?\n", "labels": "Q: Who was president when the U.S. entered World War II?\nA: Franklin D. Roosevelt."},
    {"input": "Q: Which president oversaw the end of WWII in the Pacific?\n", "labels": "Q: Which president oversaw the end of WWII in the Pacific?\nA: Harry S. Truman."},
    {"input": "Q: Who was president during the fall of the Berlin Wall (1989)?\n", "labels": "Q: Who was president during the fall of the Berlin Wall (1989)?\nA: George H. W. Bush."},
    {"input": "Q: Which president signed Medicare and Medicaid into law?\n", "labels": "Q: Which president signed Medicare and Medicaid into law?\nA: Lyndon B. Johnson."},
    {"input": "Q: Which president warned of the 'military–industrial complex'?\n", "labels": "Q: Which president warned of the 'military–industrial complex'?\nA: Dwight D. Eisenhower."},
    {"input": "Q: Who was the principal author of the Declaration of Independence and later became president?\n", "labels": "Q: Who was the principal author of the Declaration of Independence and later became president?\nA: Thomas Jefferson."},
    {"input": "Q: Which president’s doctrine promised support to countries resisting communism in 1947?\n", "labels": "Q: Which president’s doctrine promised support to countries resisting communism in 1947?\nA: Harry S. Truman (Truman Doctrine)."},
    {"input": "Q: Which president was formerly a five-star general in WWII?\n", "labels": "Q: Which president was formerly a five-star general in WWII?\nA: Dwight D. Eisenhower."}
]

In [146]:
# create the testing data
test_data = [
    {"text":"Q: Who was the 1st U.S. president?\nA:"},
    {"text":"Q: What years did Abraham Lincoln serve?\nA:"},
    {"text":"Q: Which party was Thomas Jefferson?\nA:"},
    {"text":"Q: Name the only U.S. president to resign from office.\nA:"},
    {"text":"Q: Who was president during the Louisiana Purchase?\nA:"},
    {"text":"Q: Which war occurred during James K. Polk’s presidency?\nA:"},
    {"text":"Q: What years did Franklin D. Roosevelt serve?\nA:"},
    {"text":"Q: Which president issued the Emancipation Proclamation?\nA:"},
    {"text":"Q: Which crisis in 1962 defined John F. Kennedy’s presidency?\nA:"},
    {"text":"Q: Which amendment limits presidents to two elected terms?\nA:"},
    {"text":"Q: Who was the 26th U.S. president?\nA:"},
    {"text":"Q: Which president signed the Civil Rights Act of 1964?\nA:"},
    {"text":"Q: Who was the first president to be impeached?\nA:"},
    {"text":"Q: Who was president during the Gulf War (1990–1991)?\nA:"},
    {"text":"Q: What party was Dwight D. Eisenhower?\nA:"},
]

# as well as the target data
answer_key = [
    {"text":"George Washington"},
    {"text":"1861–1865"},
    {"text":"Democratic-Republican"},
    {"text":"Richard Nixon"},
    {"text":"Thomas Jefferson"},
    {"text":"Mexican–American War"},
    {"text":"1933–1945"},
    {"text":"Abraham Lincoln"},
    {"text":"Cuban Missile Crisis"},
    {"text":"22nd Amendment"},
    {"text":"Theodore Roosevelt"},
    {"text":"Lyndon B. Johnson"},
    {"text":"Andrew Johnson"},
    {"text":"George H. W. Bush"},
    {"text":"Republican"}
]

In [147]:
import json

# save your training, testing, and target data as `jsonl` files for later usage
with open("train.jsonl", "w") as f:
    for item in train_data:
        f.write(json.dumps(item) + "\n")

with open("test.jsonl", "w") as f:
    for item in test_data:
        f.write(json.dumps(item) + "\n")

with open("target.jsonl", "w") as f:
    for item in answer_key:
        f.write(json.dumps(item) + "\n")

By saving this data in JSONL format (one JSON object per line) we can use the Hugging Face’s `datasets` library to read this dataset.

Once this data is in `jsonl` format, we can redownload it using the `load_dataset` method from HuggingFace. 

In [148]:
from datasets import load_dataset
from transformers import AutoTokenizer

# load the dataset back in 
train_dataset = load_dataset("json", data_files={"train": "train.jsonl"})
test_dataset = load_dataset("json", data_files={"test": "test.jsonl", "target": "target.jsonl"})

X_train = train_dataset["train"]
X_test = test_dataset["test"]
y_test = test_dataset["target"]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating target split: 0 examples [00:00, ? examples/s]

Just like in our `sklearn` package, we can set up multiple dataframes for different traing steps.

The `load_dataset` method creates a special `DatasetDict` object that works kind of like a dictionary with datasets inside.

We pass these datasets to their respective `X_train`, `X_test`, and `y_test` variables. We can directly access our previous data by using simple indexing.

In [149]:
X_train[0]

{'input': 'Q: Who was the 1st U.S. president?\n',
 'labels': 'Q: Who was the 1st U.S. president?\nA: George Washington.'}

In [150]:
X_test[0]

{'text': 'Q: Who was the 1st U.S. president?\nA:'}

Now that we have our data ready, let's break down our text into tokens using the `AutoTokenizer`. 

Keep in mind that when we give text to a language model, it doesn’t read characters or words directly, it intead works with tokens, which are numbers representing chunks of text.

The `AutoTokenizer` object automatically prepares our training text so that our `distilgpt2` model can interpret this text for fine-tuning.

We also set the `pad_token` attribute to guarantee that all sequences are of the same length before training. In this case we set the `pad_token` to be the "end of sentence" token (`EOS`) to indicate when one string ends, and when another begins.

In [151]:
# load the tokenizer model which will transform our text into tokens
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")

# set padding 
tokenizer.pad_token = tokenizer.eos_token

To demonstrate how this works, let's apply the tokenizer to two example sentences.

In [152]:
# example text
text1 = ["The quick brown fox"]
text2 = ["Hello world"]

# tokenize with padding and return PyTorch tensors
tokenized1 = tokenizer(text1, padding="max_length",max_length=32)

# see the numerical tokens
print("Token IDs for text1:", tokenized1["input_ids"])

Token IDs for text1: [[464, 2068, 7586, 21831, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256]]


We include the following parameters for the `Tokenizer` object:

`example["text"]` : The actual text from our dataset  
`padding="max_length"` : Pad shorter texts so all are same length  
`truncation=True` : Cut off texts that are too long  
`max_length=128` : Fixed length for all tokenized sequences  

In [153]:
# convert back to human-readable tokens
tokens = tokenizer.convert_ids_to_tokens(tokenized1["input_ids"][0])
print("Tokens:", tokens)

Tokens: ['The', 'Ġquick', 'Ġbrown', 'Ġfox', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>']


In [154]:
# tokenize with padding and return PyTorch tensors
tokenized2 = tokenizer(text2, padding="max_length",max_length=32)

# see the numerical tokens
print("Token IDs for text1:", tokenized2["input_ids"])

Token IDs for text1: [[15496, 995, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256]]


In [155]:
# convert back to human-readable tokens
tokens = tokenizer.convert_ids_to_tokens(tokenized2["input_ids"][0])
print("Tokens:", tokens)

Tokens: ['Hello', 'Ġworld', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>']


We apply this encoding to each of the sentences we will use for fine-tuning.

In [161]:
# create an empty list to store tokens for training
tokenized_texts = []

for example in train_data:
    # full sequence to learn from (prompt + answer)
    full_text = example["labels"]

    tok = tokenizer(
        full_text,
        padding="max_length",
        truncation=True,
        max_length=128
    )

    # labels should be a copy of input_ids for causal language modeling
    tok["labels"] = tok["input_ids"].copy()

    tokenized_texts.append(tok)

# peek
tokenized_texts[0]

{'input_ids': [48, 25, 5338, 373, 262, 352, 301, 471, 13, 50, 13, 1893, 30, 198, 32, 25, 4502, 2669, 13, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

Now that we've tokenized our text data we can pass it to our `Dataset.from_list()` method in order to prepare our data for training.

In [162]:
from datasets import Dataset

tokenized_dataset = Dataset.from_list(tokenized_texts)
tokenized_dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 70
})

We can finally begin setting our training parameters.

In [ ]:
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments

# import the model
model = AutoModelForCausalLM.from_pretrained("distilgpt2")

# create training arguments
training_args = TrainingArguments(
    output_dir="./finetuned",
    learning_rate=5e-5,
    per_device_train_batch_size=2,
    num_train_epochs=5,
    weight_decay=0.01,
    save_strategy="epoch"
)

Let's break down these training parameters line-by-line.

```
training_args = TrainingArguments(
    output_dir="./finetuned",
```
output_dir → folder where your fine-tuned model and checkpoints will be saved.
In this case, "./finetuned" means a folder called finetuned in the current directory.

```
    learning_rate=5e-5,
```
learning_rate → how big the model’s parameter updates are on each step.

```
    per_device_train_batch_size=2,
```
per_device_train_batch_size → how many examples are processed together per GPU/CPU in each training step.

```
    num_train_epochs=5,
```
num_train_epochs → how many times the model will see the entire training dataset.

```
    weight_decay=0.01,
```
weight_decay → a regularization technique to prevent overfitting by slightly shrinking model weights during training.

```
    save_strategy="epoch"
```
save_strategy → tells the Trainer when to save model checkpoints.

Next, we can finally begin training (note that this will take about 5 mins to train).

In [165]:
# start training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()

c:\Users\saidmf\anaconda3\envs\ds\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


c:\Users\saidmf\anaconda3\envs\ds\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\saidmf\anaconda3\envs\ds\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\saidmf\anaconda3\envs\ds\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\saidmf\anaconda3\envs\ds\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=175, training_loss=0.04703810010637556, metrics={'train_runtime': 344.8633, 'train_samples_per_second': 1.015, 'train_steps_per_second': 0.507, 'total_flos': 11431732838400.0, 'train_loss': 0.04703810010637556, 'epoch': 5.0})

In [166]:
# save model
trainer.save_model("./finetuned")
# save tokenizer
tokenizer.save_pretrained("./finetuned")

('./finetuned\\tokenizer_config.json',
 './finetuned\\special_tokens_map.json',
 './finetuned\\vocab.json',
 './finetuned\\merges.txt',
 './finetuned\\added_tokens.json',
 './finetuned\\tokenizer.json')

Now that we have our model trained, let's evaluate the models responses and how closely these outputs match to our anticipated outputs.

First, we will iterate through all of our training data and generate predictions.

In [167]:
import torch

# generate predictions
yhat = []
for i in range(len(X_test)):
    prompt = X_test[i]["text"]
    target = y_test[i]["text"]

    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    answer_text = full_text.split("A:", 1)[-1].strip()
    answer_text = answer_text.split("\n", 1)[0].strip()

    yhat.append({"prompt": prompt, "prediction": answer_text, "target": target})

yhat

[{'prompt': 'Q: Who was the 1st U.S. president?\nA:',
  'prediction': 'Andrew Jackson.',
  'target': 'George Washington'},
 {'prompt': 'Q: What years did Abraham Lincoln serve?\nA:',
  'prediction': '1861–1865.',
  'target': '1861–1865'},
 {'prompt': 'Q: Which party was Thomas Jefferson?\nA:',
  'prediction': 'Republican.',
  'target': 'Democratic-Republican'},
 {'prompt': 'Q: Name the only U.S. president to resign from office.\nA:',
  'prediction': 'Gerald Ford.',
  'target': 'Richard Nixon'},
 {'prompt': 'Q: Who was president during the Louisiana Purchase?\nA:',
  'prediction': 'Thomas Jefferson.',
  'target': 'Thomas Jefferson'},
 {'prompt': 'Q: Which war occurred during James K. Polk’s presidency?\nA:',
  'prediction': 'Mexican-American War.',
  'target': 'Mexican–American War'},
 {'prompt': 'Q: What years did Franklin D. Roosevelt serve?\nA:',
  'prediction': '1933–1945.',
  'target': '1933–1945'},
 {'prompt': 'Q: Which president issued the Emancipation Proclamation?\nA:',
  'pred

We can evaluate predictions through a variety of means. one simple way is to just calculate the ratio of responses that are correct.

In [168]:
total_correct = 0

for pred in yhat:
    if pred["target"] in pred["prediction"] or pred["prediction"] in pred["target"]:
        total_correct += 1

accurate_rate = total_correct / len(yhat)
print(f"Total Accuracy: {accurate_rate * 100}%")

Total Accuracy: 40.0%


Let's also test this model out on a single example to see how this output has influenced our model!

In [169]:
generator = pipeline("text-generation", model="./finetuned", tokenizer="./finetuned")

prompt = "Q: Who was the 35th U.S. president? \nA:"

generator(prompt, max_length=10, do_sample=True)

Device set to use cpu
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Both `max_new_tokens` (=256) and `max_length`(=10) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'Q: Who was the 35th U.S. president? \nA:. president?\nA: Gerald Ford.'}]

## Fine-Tuning in OpenAI

Again it's going to take a lot more training to get `distilgpt2` to a level where it does not use language like "getting iced out." Therefore, for the remainder of this exercise we will, again, utilize the `openai` API. 

We will pass modified training data to manipulate `gpt-4.1-mini`'s answers.

To begin running the code below, copy & paste the provided API key in the code-block below.

**DO NOT PUSH THIS KEY TO GITHUB**  
**DO NOT PUSH THIS KEY TO GITHUB**  
**DO NOT PUSH THIS KEY TO GITHUB**  

In [170]:
api_key = "..."

In [ ]:
from openai import OpenAI

# access the specific OpenAI project
client = OpenAI(api_key="...", project="proj_fHRnVJY0Oyfm1ufG1sffxa6W")

In [172]:
# prepare the pres_facts jsonl file for fine-tuning
upload = client.files.create(file=open("pres_facts.jsonl","rb"), purpose="fine-tune")

In [ ]:
# TODO: begin fine-tuning the gpt-4.1-nano model on our input data (given to one person)
...

**Wait until fine-tuning is complete** (best to have one person do this, as this will take about 10 minutes to complete).

In [ ]:
j = client.fine_tuning.jobs.retrieve(job.id)
print(j.status)

Once the above is "succeeded", you can run the next batch of code to evaluate how your fine-tuned model's responses.

In [183]:
ft_model = "ft:gpt-4.1-nano-2025-04-14:lusitania::C3nnZhc2"

resp = client.responses.create(
    model=ft_model,
    input=prompt
)

print(resp.output_text)

John F. Kennedy.


## Unpredictability in Fine-Tuned Models - Interesting Research

While this fine-tuned model is limited and will not lead to emergent behavior, typically tuned models also have "emergent" or "unpredictable" behavior.

This space is ripe for innovation and exploration. Check out the following articles to find out more how fine-tuning changes model behavior.

* https://techcrunch.com/2025/07/10/grok-4-seems-to-consult-elon-musk-to-answer-controversial-questions/ 
* https://arxiv.org/html/2502.17424v1